# Introduction

In this notebook, we present data analysis on Chatbot Arena data collected from https://arena.lmsys.org.

We explain different Elo calculation methods (online Elo and Bradley-Terry model) for model ranking.

To view the latest leaderboard, see https://huggingface.co/spaces/lmsys/chatbot-arena-leaderboard.


In [3]:
from collections import defaultdict
import json, math
# , gdown
import numpy as np
import pandas as pd
# import plotly.express as px
from tqdm import tqdm
import requests
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.plot_util import *

# Load Historical Data from HF

In [5]:
# Import datasets from https://huggingface.co/datasets/lmarena-ai/arena-human-preference-55k
from datasets import load_dataset
ds = load_dataset("lmarena-ai/arena-human-preference-55k")

In [6]:
# inspect the available splits
print(ds)  
# grab the ‘train’ split
train = ds["train"]

DatasetDict({
    train: Dataset({
        features: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie'],
        num_rows: 57477
    })
})


In [7]:
df = train.to_pandas()
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [10]:
# Create 'result' column
df['winner'] = df.apply(
    lambda row: 'tie' if row['winner_tie'] == 1 else (
                'model_b' if row['winner_model_b'] == 1 else 'model_a'),
    axis=1
)

## Code to produce ELO rankings, taken from LLMArena Notebook

### [Bradley-Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model)

In the context of LLM evaluation, models can be assumed to be static. In this case, we can directly fit the ratings with Bradley-Terry model, which produce significantly stable ratings. Here we provide an implementation with logistic regression.

In [19]:
def compute_mle_elo(
    df, SCALE=400, BASE=10, INIT_RATING=1000, sample_weight=None
):
    from sklearn.linear_model import LogisticRegression
    ptbl_a_win = pd.pivot_table(
        df[df["winner"] == "model_a"],
        index="model_a",
        columns="model_b",
        aggfunc="size",
        fill_value=0,
    )
    # if no tie, create a zero matrix
    if sum(df["winner"].isin(["tie", "tie (bothbad)"])) == 0:
        ptbl_tie = pd.DataFrame(0, index=ptbl_a_win.index, columns=ptbl_a_win.columns)
    else:
        ptbl_tie = pd.pivot_table(
            df[df["winner"].isin(["tie", "tie (bothbad)"])],
            index="model_a",
            columns="model_b",
            aggfunc="size",
            fill_value=0,
        )
        ptbl_tie = ptbl_tie + ptbl_tie.T

    ptbl_b_win = pd.pivot_table(
        df[df["winner"] == "model_b"],
        index="model_a",
        columns="model_b",
        aggfunc="size",
        fill_value=0,
    )
    ptbl_win = ptbl_a_win * 2 + ptbl_b_win.T * 2 + ptbl_tie

    models = pd.Series(np.arange(len(ptbl_win.index)), index=ptbl_win.index)

    p = len(models)
    X = np.zeros([p * (p - 1) * 2, p])
    Y = np.zeros(p * (p - 1) * 2)

    cur_row = 0
    sample_weights = []
    for m_a in ptbl_win.index:
        for m_b in ptbl_win.columns:
            if m_a == m_b:
                continue
            # if nan skip
            if math.isnan(ptbl_win.loc[m_a, m_b]) or math.isnan(ptbl_win.loc[m_b, m_a]):
                continue
            X[cur_row, models[m_a]] = +math.log(BASE)
            X[cur_row, models[m_b]] = -math.log(BASE)
            Y[cur_row] = 1.0
            sample_weights.append(ptbl_win.loc[m_a, m_b])

            X[cur_row + 1, models[m_a]] = math.log(BASE)
            X[cur_row + 1, models[m_b]] = -math.log(BASE)
            Y[cur_row + 1] = 0.0
            sample_weights.append(ptbl_win.loc[m_b, m_a])
            cur_row += 2
    X = X[:cur_row]
    Y = Y[:cur_row]

    lr = LogisticRegression(fit_intercept=False, penalty=None, tol=1e-6)
    lr.fit(X, Y, sample_weight=sample_weights)
    elo_scores = SCALE * lr.coef_[0] + INIT_RATING
    if "mixtral-8x7b-instruct-v0.1" in models.index:
        elo_scores += 1114 - elo_scores[models["mixtral-8x7b-instruct-v0.1"]]
    return pd.Series(elo_scores, index=models.index).sort_values(ascending=False)

In [ ]:
def preety_print_model_ratings(ratings):
    df = pd.DataFrame([
        [n, ratings[n]] for n in ratings.keys()
    ], columns=["Model", "Elo rating"]).sort_values("Elo rating", ascending=False).reset_index(drop=True)
    # df["Elo rating"] = (df["Elo rating"] + 0.5).astype(int)
    df.index = df.index + 1
    return df

## Run Robustness Check on Full Arena Data (including ties)

In [42]:
rawBT_Ties = df[['model_a', 'model_b', 'winner_model_a']]
rawBT_Ties.head()

,model_a,model_b,winner_model_a
0,gpt-4-1106-preview,gpt-4-0613,1
1,koala-13b,gpt-4-0613,0
2,gpt-3.5-turbo-0613,mistral-medium,0
3,llama-2-13b-chat,mistral-7b-instruct,1
4,koala-13b,gpt-3.5-turbo-0314,0


In [44]:
# make design matrix for BT.
X, y, player_to_id = make_BT_design_matrix(rawBT_Ties)
X.shape, y.shape

((57477, 63), (57477,))

In [ ]:
# PERFORM ROBUSTNESS AUDITING ON FULL ARENA DATA (INCLUDING TIES).
# YOU CAN SKIP THIS CELL (AND THE ONE BELOW) IF YOU ALREADY HAVE THE ROBUSTNESS AUDITING RESULTS 
# (AKA ELOChatbotArenaNonrobust.pkl.)
ks = [1, 3, 5, 10, 20]
results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [60]:
# find the (k, alpha N) pairs that are top-k non-robust.
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

{(1, 2): (21,
  None,
  -0.006288794711674339,
  0.0004984131512624942,
  array([57339, 33872])),
 (3, 24): (6,
  41,
  0.12679781444443133,
  -0.004115836757440372,
  array([12742, 35121, 28919,  3712, 42442, 32268, 55829, 52352, 20172,
         17355, 25481, 25267, 36203, 35393,  2396, 13244,  3483, 40406,
         22020, 39307,  6887, 22343, 37129, 44925])),
 (5, 16): (41,
  47,
  0.07868100815171142,
  -0.0018388413128466174,
  array([13835, 23088, 56334,  9842, 50133, 38755, 52631, 27552, 55511,
         29584, 49513, 49509, 54538, 14102, 20425, 39862])),
 (10, 3): (37,
  42,
  0.008225282137498291,
  -0.003423033226177541,
  array([11865,  4026, 19049])),
 (20, 1): (48,
  29,
  0.008398628526396834,
  -0.0006804081990048338,
  array([3964]))}

In [ ]:
import pickle
# with open('ELOChatbotArenaNonrobust.pkl', 'wb') as f:
#     pickle.dump(results_nonrobust, f)

# Load the pickle file
with open('results/ELOChatbotArenaNonrobust.pkl', 'rb') as f:
    ELOChatbotArenaNonrobust = pickle.load(f)

In [ ]:
# format results to inspect dropped matches.
old_tuple = results_nonrobust[(1, 2)] # = 0 # change none to -1 (gpt-4-1106-preview).
new_tuple = tuple(-1 if i == 1 and val is None else val for i, val in enumerate(old_tuple))
results_nonrobust[(1, 2)] = new_tuple

rows = []
for (k, aN), (playerA, playerB, original_beta_diff, new_beta_diff_refit, indices) in results_nonrobust.items():
    rows.append({
        "k-aN": (k, aN),
        "playerA": playerA + 1, # to account for the reference index.
        "playerB": playerB + 1,
        "original_beta_diff": original_beta_diff,
        "new_beta_diff_refit": new_beta_diff_refit,
        "indices": indices
    })
cba_results = pd.DataFrame(rows)
cba_results.head()

In [ ]:
# reverse the player to id dictionary for easy identification of flipped models.
id_to_player = {v: k for k, v in player_to_id.items()}
id_to_player[7], id_to_player[42]

### Compare original vs. new rankings (plain BT score version)
aka, logistic regression coefficients.

In [ ]:
k, alphaN = 1, 2
playerA, playerB, orig_out, new_out, indices = ELOChatbotArenaNonrobust[k, alphaN]

In [147]:
model_full = run_logistic_regression(X, y)
Xd = np.delete(X, indices, axis=0)
yd = np.delete(y, indices, axis=0)
model_d  = run_logistic_regression(Xd, yd)

# prepend model 0, the reference model, which has score 0.
orig_scores = np.insert(model_full.coef_[0], 0, 0)
print("scores on full data: ", orig_scores)
new_scores = np.insert(model_d.coef_[0], 0, 0)
print("scores after dropping AMIS: ", new_scores)

# get play_id, orig_score
indexed_orig_scores = list(enumerate(orig_scores))
indexed_new_scores = list(enumerate(new_scores)) 
# get player_id, sorted_new_score 
sorted_original_scores = sorted(indexed_orig_scores, key=lambda x: x[1], reverse=True)
print("sorted scores on full data: ", sorted_original_scores)
sorted_new_scores = sorted(indexed_new_scores, key=lambda x: x[1], reverse=True)
print("sorted scores after dropping AMIS: ", sorted_new_scores)


scores on full data:  [ 0.         -1.61682693 -0.83804266 -1.14811563 -1.27965413 -0.78902603
 -0.7902158  -0.42467409 -1.48217007 -1.73720294 -1.04401262 -1.31322306
 -1.07411807 -0.64880127 -0.80998895 -2.23811127 -0.8878999  -2.5945846
 -0.75027457 -1.08790256 -1.030317   -0.84446126 -0.00628879 -1.1809007
 -0.52659174 -0.86680562 -2.53780256 -0.86149359 -2.04497667 -1.16857894
 -0.87508771 -2.62797281 -1.4364421  -1.15812092 -1.18498194 -0.76773911
 -0.85168633 -1.08913623 -0.77994837 -2.07892891 -1.32980913 -2.0519981
 -0.55147191 -0.78817366 -1.14985902 -1.37973905 -2.15213788 -1.03760502
 -0.63015292 -0.86668908 -1.20903025 -1.050682   -1.49918081 -0.96069183
 -1.97584768 -1.40756866 -1.14915163 -1.99585158 -1.26217447 -1.4364132
 -1.89716622 -1.37957347 -1.08334353 -0.94630852]
scores after dropping AMIS:  [ 0.         -1.61724579 -0.83847198 -1.14855701 -1.28008676 -0.78950245
 -0.79056394 -0.4250637  -1.48258902 -1.73760377 -1.04442621 -1.31364281
 -1.07452901 -0.64920612 -0

In [ ]:
# each tuple holds: llm name, llm id, original score, score after dropping the AMIS.
# the ordering follows the original scores in descending order.
rankings = return_rankings_list(X, y, results, 3, 24, player_to_id)
rankings

[['gpt-4-1106-preview', 0, 0.0, 0.0],
 ['gpt-4-0125-preview', 22, -0.006288794711674339, -0.004864421658799411],
 ['gpt-4-0314', 7, -0.4246740945498912, -0.4258369030283142],
 ['gpt-4-0613', 24, -0.5265917430538124, -0.5271276977785239],
 ['qwen1.5-72b-chat', 42, -0.5514719089943225, -0.4217210662708298],
 ['mistral-medium', 48, -0.6301529171460339, -0.6290896145156049],
 ['claude-1', 13, -0.6488012720962829, -0.6503429307982282],
 ['claude-2.0', 18, -0.7502745691278374, -0.7520784820369764],
 ['gemini-pro-dev-api', 35, -0.7677391067625561, -0.767605634881177],
 ['yi-34b-chat', 38, -0.7799483738803993, -0.7784895460671264],
 ['gpt-3.5-turbo-0125', 43, -0.7881736560178976, -0.7770471662363982],
 ['mixtral-8x7b-instruct-v0.1', 5, -0.7890260304078561, -0.7878769681955522],
 ['gemini-pro', 6, -0.7902157974680525, -0.7913226339839818],
 ['claude-2.1', 14, -0.8099889487631253, -0.8108389320243099],
 ['gpt-3.5-turbo-0613', 2, -0.8380426595431233, -0.8393156173636073],
 ['starling-lm-7b-alpha'

### Compare original vs. new rankings
these ELO scores are computed using the ELO computation in the LMArena Notebook.

In [146]:
elo_mle_ratings = compute_mle_elo(df)
bt_elo_table = preety_print_model_ratings(elo_mle_ratings)
bt_elo_table.head(10)

,Model,Elo rating
1,gpt-4-1106-preview,1251.06
2,gpt-4-0125-preview,1250.42
3,gpt-4-0314,1178.82
4,gpt-4-0613,1160.35
5,qwen1.5-72b-chat,1148.42
6,mistral-medium,1145.99
7,claude-1,1138.41
8,claude-2.0,1123.65
9,gemini-pro-dev-api,1116.89
10,gemini-pro,1114.13


ELO scores after removing the AMIS.

In [141]:
# drop MIS.
top_1_indices = cba_results['indices'][0]
top_3_indices = cba_results['indices'][1]
top_5_indices = cba_results['indices'][2]
top_10_indices = cba_results['indices'][3]
top_20_indices = cba_results['indices'][4]

newdf_top1 = df.drop(index=top_1_indices)
newdf_top3 = df.drop(index=top_3_indices)
newdf_top5 = df.drop(index=top_5_indices)
newdf_top10 = df.drop(index=top_10_indices)
newdf_top20 = df.drop(index=top_20_indices)

In [144]:
elo_mle_ratings_newdf_top1 = compute_mle_elo(newdf_top1)
bt_elo_table_newdf_top1 = preety_print_model_ratings(elo_mle_ratings_newdf_top1)
print(bt_elo_table_newdf_top1.head(10))

elo_mle_ratings_newdf_top3 = compute_mle_elo(newdf_top3)
bt_elo_table_newdf_top3 = preety_print_model_ratings(elo_mle_ratings_newdf_top3)
print(bt_elo_table_newdf_top3.head(10))

elo_mle_ratings_newdf_top5 = compute_mle_elo(newdf_top5)
bt_elo_table_newdf_top5 = preety_print_model_ratings(elo_mle_ratings_newdf_top5)
print(bt_elo_table_newdf_top5.head(10))

elo_mle_ratings_newdf_top10 = compute_mle_elo(newdf_top10)
bt_elo_table_newdf_top10 = preety_print_model_ratings(elo_mle_ratings_newdf_top10)
print(bt_elo_table_newdf_top10.head(20))

elo_mle_ratings_newdf_top20 = compute_mle_elo(newdf_top20)
bt_elo_table_newdf_top20 = preety_print_model_ratings(elo_mle_ratings_newdf_top20)
print(bt_elo_table_newdf_top20.head(20))

                 Model  Elo rating
1   gpt-4-1106-preview     1251.08
2   gpt-4-0125-preview     1250.89
3           gpt-4-0314     1178.84
4           gpt-4-0613     1160.30
5     qwen1.5-72b-chat     1148.43
6       mistral-medium     1146.01
7             claude-1     1138.43
8           claude-2.0     1123.64
9   gemini-pro-dev-api     1116.92
10          gemini-pro     1114.15
                         Model  Elo rating
1           gpt-4-1106-preview     1250.92
2           gpt-4-0125-preview     1250.44
3                   gpt-4-0314     1178.58
4             qwen1.5-72b-chat     1163.34
5                   gpt-4-0613     1160.14
6               mistral-medium     1145.97
7                     claude-1     1138.13
8                   claude-2.0     1123.33
9           gemini-pro-dev-api     1116.79
10  mixtral-8x7b-instruct-v0.1     1114.00
                 Model  Elo rating
1   gpt-4-1106-preview     1252.08
2   gpt-4-0125-preview     1250.47
3           gpt-4-0314     1179.03
4 

### Manuel check (for each k) on the MIS

inspect the indices in the MIS. they do indeed consist of matchups that contain the LLMs that flipped rankings.

In [ ]:
# for each index, find the corresponding row in the original dataframe.
indices = top_1_indices
df.iloc[indices]

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,winner
57339,4284047809,gpt-4-0125-preview,chatglm3-6b,"[""Start the game. Ask the user for their grade...","[""What is your grade level?"",""The word is \""ma...","[""Sure, I'd be happy to play this game with yo...",0,0,1,tie
33872,2516736149,gpt-4-0125-preview,chatglm3-6b,"[""You are a helpful and honest assistant for c...","[""{\""Bulk message\"":\""\"", \""Urgency and fear t...","[""{\n\""Bulk message\"":\""no\"",\n\""Urgency and f...",0,0,1,tie


In [ ]:
mis_inds = cba_results['indices'][0]
X_new = np.delete(X, mis_inds, axis=0)
y_new = np.delete(y, mis_inds, axis=0)

res_full = run_logistic_regression(X,
                                    y,
                                    fit_intercept=False, 
                                    penalty=None
                                    )
res_dropped = run_logistic_regression(X_new,
                                y_new,
                                fit_intercept=False, 
                                penalty=None
                                )

In [ ]:
player1 = cba_results['playerA'][0] - 1
player2 = cba_results['playerB'][0] - 1
beta_diff = res_full.coef_[0][player1] - res_full.coef_[0][player2]

# difference in betas between the players that flipped rankings 
# (before and after dropping the AMIS).
beta_diff_refit = res_dropped.coef_[0][player1] - res_dropped.coef_[0][player2]
beta_diff, beta_diff_refit

# res_full.coef_[0][player1], res_dropped.coef_[0][player1] # use this line if one of the players is "None".

(0.12679781444443133, -0.004115836757484392)

In [117]:
# 3rd place, 5th place (full data)
res_full.coef_[0][player1], res_full.coef_[0][player2]

(-0.4246740945498912, -0.5514719089943225)

Below is code from the original LLMArena Notebook.

### Compute Bootstrap Confidence Interavals for BT Scores

We can further use bootstrap to estimate the confidence intervals as well.


In [ ]:
def get_bootstrap_result(battles, func_compute_elo, num_round):
    rows = []
    for i in tqdm(range(num_round), desc="bootstrap"):
        rows.append(func_compute_elo(battles.sample(frac=1.0, replace=True)))
    df = pd.DataFrame(rows)
    return df[df.median().sort_values(ascending=False).index]


In [ ]:
BOOTSTRAP_ROUNDS = 100

np.random.seed(42)
bootstrap_elo_lu = get_bootstrap_result(battles, compute_mle_elo, BOOTSTRAP_ROUNDS)

bootstrap: 100%|██████████| 100/100 [15:04<00:00,  9.04s/it]


In [ ]:
def visualize_bootstrap_scores(df, title):
    bars = pd.DataFrame(dict(
        lower = df.quantile(.025),
        rating = df.quantile(.5),
        upper = df.quantile(.975))).reset_index(names="model").sort_values("rating", ascending=False)
    bars['error_y'] = bars['upper'] - bars["rating"]
    bars['error_y_minus'] = bars['rating'] - bars["lower"]
    bars['rating_rounded'] = np.round(bars['rating'], 2)
    fig = px.scatter(bars, x="model", y="rating", error_y="error_y",
                     error_y_minus="error_y_minus", text="rating_rounded",
                     title=title)
    fig.update_layout(xaxis_title="Model", yaxis_title="Rating",
                      height=600)
    return fig

fig = visualize_bootstrap_scores(bootstrap_elo_lu, "Bootstrap of MLE Elo Rating Estimates")
fig

We previously apply bootstrapping on the online Elo to obtain stabler ratings.

In [ ]:
np.random.seed(42)
bootstrap_online_elo = get_bootstrap_result(battles, compute_online_elo, BOOTSTRAP_ROUNDS)

bootstrap: 100%|██████████| 100/100 [14:31<00:00,  8.72s/it]


We can see the bootstrapping medians obtained by both methods are similar.

In [ ]:
preety_print_two_ratings(bootstrap_elo_lu.quantile(.5),
                         bootstrap_online_elo.quantile(.5),
                         column_names=["Bootstrap Median of BT", "Bootstrap Median of Online Elo"])

,Model,Bootstrap Median of MLE Elo,Bootstrap Median of Online Elo
1,chatgpt-4o-latest,1315,1315
2,gemini-1.5-pro-exp-0801,1298,1295
3,gpt-4o-2024-05-13,1286,1288
4,gpt-4o-mini-2024-07-18,1274,1278
5,claude-3-5-sonnet-20240620,1271,1272
...,...,...,...
125,chatglm-6b,879,881
126,fastchat-t5-3b,868,870
127,stablelm-tuned-alpha-7b,839,839
128,dolly-v2-12b,822,822


However, online Elo's confidence intervals are significantly larger than the BT.

In [ ]:
fig = visualize_bootstrap_scores(bootstrap_online_elo, "Bootstrap of Online Elo Rating Estimates")
fig

### Predict Win Rates
Utilizing Elo ratings allows us to predict win probabilities. By comparing the predicted win rates with the actual win rates, we can gain insight into the accuracy and quality of the Elo rating system.






In [ ]:
def predict_win_rate(elo_ratings, SCALE=400, BASE=10, INIT_RATING=1000):
    names = sorted(list(elo_ratings.keys()))
    wins = defaultdict(lambda: defaultdict(lambda: 0))
    for a in names:
        for b in names:
            ea = 1 / (1 + BASE ** ((elo_ratings[b] - elo_ratings[a]) / SCALE))
            wins[a][b] = ea
            wins[b][a] = 1 - ea

    data = {
        a: [wins[a][b] if a != b else np.NAN for b in names]
        for a in names
    }

    df = pd.DataFrame(data, index=names)
    df.index.name = "model_a"
    df.columns.name = "model_b"
    return df.T

In [ ]:
win_rate = predict_win_rate(dict(bootstrap_elo_lu.quantile(0.5)))
ordered_models = win_rate.mean(axis=1).sort_values(ascending=False).index
ordered_models = ordered_models[:30]
fig = px.imshow(win_rate.loc[ordered_models, ordered_models],
                color_continuous_scale='RdBu', text_auto=".2f",
                title="Predicted Win Rate Using Elo Ratings for Model A in an A vs. B Battle")
fig.update_layout(xaxis_title="Model B",
                  yaxis_title="Model A",
                  xaxis_side="top", height=900, width=900,
                  title_y=0.07, title_x=0.5)
fig.update_traces(hovertemplate=
                  "Model A: %{y}<br>Model B: %{x}<br>Win Rate: %{z}<extra></extra>")
fig

### Compute Bootstrap Confidence Intervals Assuming Uniform Sampling

We also study how the ratings will change if we only sample an equal number of battles for each model pair.

In [ ]:
def sample_battle_even(battles, n_per_battle):
    groups = battles.groupby(["model_a", "model_b"], as_index=False)
    resampled = (groups
                 .apply(lambda grp: grp.sample(n_per_battle, replace=True))
                 .reset_index(drop=True))
    return resampled

In [ ]:
num_samples = 50
battles_even = sample_battle_even(battles, num_samples)
pd.pivot_table(battles_even, index="model_a", columns="model_b", aggfunc="size", fill_value=0)

model_b,RWKV-4-Raven-14B,alpaca-13b,athene-70b-0725,bard-jan-24-gemini-pro,chatglm-6b,chatglm2-6b,chatglm3-6b,chatgpt-4o-latest,claude-1,claude-2.0,...,vicuna-7b,wizardlm-13b,wizardlm-70b,yi-1.5-34b-chat,yi-34b-chat,yi-large,yi-large-preview,zephyr-7b-alpha,zephyr-7b-beta,zephyr-orpo-141b-A35b-v0.1
model_a,,,,,,,,,,,,,,,,,,,,,
RWKV-4-Raven-14B,0,50,0,0,50,0,0,0,50,50,...,50,50,0,0,0,0,0,0,0,0
alpaca-13b,50,0,0,0,50,0,0,0,50,50,...,50,50,0,0,0,0,0,0,0,0
athene-70b-0725,0,0,0,0,0,0,0,50,0,0,...,0,0,0,0,0,50,50,0,0,0
bard-jan-24-gemini-pro,0,0,0,0,0,0,50,0,50,50,...,0,0,50,0,50,0,0,0,50,0
chatglm-6b,50,50,0,0,0,0,0,0,50,50,...,50,50,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
yi-large,0,0,50,0,0,0,0,0,0,0,...,0,0,0,50,0,0,50,0,0,0
yi-large-preview,0,0,50,0,0,0,0,0,0,0,...,0,0,0,50,50,50,0,0,0,0
zephyr-7b-alpha,0,0,0,0,0,50,0,0,50,50,...,50,50,50,0,0,0,0,0,50,0


In [ ]:
# Sampling Battles Evenly
def get_bootstrap_even_sample(battles, n_per_battle, func_compute_elo, num_round=BOOTSTRAP_ROUNDS):
    rows = []
    for n in tqdm(range(num_round), desc="sampling battles evenly"):
        resampled = sample_battle_even(battles, n_per_battle)
        rows.append(func_compute_elo(resampled))
    df = pd.DataFrame(rows)
    return df[df.median().sort_values(ascending=False).index]

In [ ]:
print("number of samples per battle pair:", num_samples)
bootstrap_even_lu = get_bootstrap_even_sample(battles, num_samples, compute_mle_elo, num_round=100)

number of samples per battle pair: 50


sampling battles evenly: 100%|██████████| 100/100 [16:29<00:00,  9.90s/it]


In [ ]:
fig = visualize_bootstrap_scores(bootstrap_even_lu, f"Bootstrap of MLE Elo Estimates - Even sample")
fig

# Language-specific Leaderboards
We present two language-specific leaderboards, by isolating the chat data into two subsets based on the language: (1) English-only and (2) Non-English.

## English-only

In [ ]:
english_only_battles = battles[battles["language"] == "English"]
elo_ratings = compute_mle_elo(english_only_battles)
pd.DataFrame(elo_ratings)

,0
model_a,
chatgpt-4o-latest,1294.00
gemini-1.5-pro-exp-0801,1266.55
gpt-4o-2024-05-13,1263.66
llama-3.1-405b-instruct,1256.89
gpt-4o-mini-2024-07-18,1255.61
...,...
fastchat-t5-3b,871.25
chatglm-6b,858.01
stablelm-tuned-alpha-7b,830.02


## Non-English

In [ ]:
non_english_battles = battles[battles["language"] != "English"]
elo_ratings = compute_mle_elo(non_english_battles)
pd.DataFrame(elo_ratings)

,0
model_a,
chatgpt-4o-latest,1349.52
gemini-1.5-pro-exp-0801,1344.29
gpt-4o-2024-05-13,1319.01
gemini-advanced-0514,1309.93
claude-3-5-sonnet-20240620,1308.97
...,...
oasst-pythia-12b,880.52
dolly-v2-12b,860.78
llama-13b,848.49


# Links



Some good resources to learn more about Elo rating systems:
- Elo rating system https://en.wikipedia.org/wiki/Elo_rating_system
- Bradley-Terry model https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model
- An introduction video https://www.youtube.com/watch?v=AsYfbmp0To0
- A FiveThirtyEight article https://fivethirtyeight.com/methodology/how-our-nfl-predictions-work/
